In [1]:
# cell: 1
import os 
import certifi
import requests
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_classic import hub

from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder



C:\Users\Omar Faruk Jihad\AppData\Local\Temp\ipykernel_16200\1914670802.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


In [2]:
# cell: 2
from langchain_classic.agents import create_react_agent, AgentExecutor

In [3]:
# cell: 3

os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
WEATHER_API_KEY = os.getenv("WEATHER_API_KEY")

if not GOOGLE_API_KEY:
  raise ValueError("Google API not found!")


In [4]:
# cell: 4
search_tool = TavilySearchResults(max_results=2)

C:\Users\Omar Faruk Jihad\AppData\Local\Temp\ipykernel_16200\3783858571.py:2: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(max_results=2)


In [5]:
def fetch_weather(city: str):

    # Step 1: Get city coordinates
    geo_url = "https://geocoding-api.open-meteo.com/v1/search"

    geo_params = {
        "name": city,
        "count": 1,
        "language": "en",
        "format": "json"
    }

    geo_response = requests.get(
        geo_url,
        params=geo_params,
        timeout=15
    )

    geo_response.raise_for_status()

    geo_data = geo_response.json()

    if not geo_data.get("results"):
        return f"City not found: {city}"

    location = geo_data["results"][0]

    latitude = location["latitude"]
    longitude = location["longitude"]

    # Step 2: Get weather data
    weather_url = "https://api.open-meteo.com/v1/forecast"

    weather_params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": (
            "temperature_2m,"
            "relative_humidity_2m,"
            "apparent_temperature,"
            "weather_code,"
            "wind_speed_10m,"
            "precipitation"
        ),
        "timezone": "auto"
    }

    weather_response = requests.get(
        weather_url,
        params=weather_params,
        timeout=15
    )

    weather_response.raise_for_status()

    weather_data = weather_response.json()

    current = weather_data["current"]

    # Step 3: Weather code descriptions
    weather_codes = {
        0: "Clear sky",
        1: "Mainly clear",
        2: "Partly cloudy",
        3: "Overcast",
        45: "Fog",
        48: "Depositing rime fog",
        51: "Light drizzle",
        53: "Moderate drizzle",
        55: "Dense drizzle",
        56: "Freezing drizzle",
        57: "Dense freezing drizzle",
        61: "Slight rain",
        63: "Moderate rain",
        65: "Heavy rain",
        66: "Light freezing rain",
        67: "Heavy freezing rain",
        71: "Slight snowfall",
        73: "Moderate snowfall",
        75: "Heavy snowfall",
        77: "Snow grains",
        80: "Slight rain showers",
        81: "Moderate rain showers",
        82: "Violent rain showers",
        85: "Slight snow showers",
        86: "Heavy snow showers",
        95: "Thunderstorm",
        96: "Thunderstorm with slight hail",
        99: "Thunderstorm with heavy hail"
    }

    weather_description = weather_codes.get(
        current["weather_code"],
        "Unknown weather condition"
    )

    # Step 4: Return weather information
    return {
        "city": location["name"],
        "country": location.get("country", "Unknown"),
        "temperature": current["temperature_2m"],
        "feels_like": current["apparent_temperature"],
        "humidity": current["relative_humidity_2m"],
        "wind_speed": current["wind_speed_10m"],
        "precipitation": current["precipitation"],
        "weather": weather_description,
        "time": current["time"]
    }


print("Open-Meteo Weather Function created!")

Open-Meteo Weather Function created!


In [6]:
weather = fetch_weather("Dhaka")

print("Weather Information")
print("-------------------")

for key, value in weather.items():
    print(f"{key}: {value}")

Weather Information
-------------------
city: Dhaka
country: Bangladesh
temperature: 24.9
feels_like: 27.3
humidity: 74
wind_speed: 9.1
precipitation: 0.0
weather: Clear sky
time: 2026-11-24T09:30


In [7]:
@tool
def get_weather_data(city: str):
    """Get current weather information for any city using Open-Meteo API."""

    try:
        weather = fetch_weather(city)

        if isinstance(weather, str):
            return weather

        return (
            f"City: {weather['city']}\n"
            f"Country: {weather['country']}\n"
            f"Temperature: {weather['temperature']}°C\n"
            f"Feels Like: {weather['feels_like']}°C\n"
            f"Humidity: {weather['humidity']}%\n"
            f"Wind Speed: {weather['wind_speed']} km/h\n"
            f"Precipitation: {weather['precipitation']} mm\n"
            f"Weather: {weather['weather']}\n"
            f"Local Time: {weather['time']}"
        )

    except requests.exceptions.RequestException as e:
        return f"Weather API connection error: {str(e)}"

    except (KeyError, ValueError, TypeError) as e:
        return f"Weather data processing error: {str(e)}"


print("Open-Meteo Weather Tool created!")

Open-Meteo Weather Tool created!


In [8]:
result = get_weather_data.invoke({
    "city": "Dhaka"
})

print(result)

City: Dhaka
Country: Bangladesh
Temperature: 24.9°C
Feels Like: 27.3°C
Humidity: 74%
Wind Speed: 9.1 km/h
Precipitation: 0.0 mm
Weather: Clear sky
Local Time: 2026-11-24T09:30


In [9]:
# cell: 5
result = search_tool.invoke("What is the capital of Bnagladesh?")
result

[{'title': 'Dhaka - Wikipedia',
  'url': 'https://en.wikipedia.org/wiki/Dhaka',
  'content': "Dhaka (/ˈdɑːkə/ⓘ DAH-kə or /ˈdækə/ⓘ DAK-ə; Bengali: ঢাকা, romanized:Ḍhākā, pronounced\\[ˈɖʱaka]ⓘ), formerly spelled as Dacca,\\ is the capital and largest city of Bangladesh. With an estimated population of 36.6 million, Dhaka is the second largest city by population in the world,\\\\ and is widely considered to be the most densely populated built-up urban area in the world.\\\\ Dhaka is an important cultural, economic, and scientific hub of Eastern South Asia. Dhaka ranks fourth in South Asia and 55th in the world in terms of GDP. Lying on the Ganges Delta, it is bounded by the Buriganga, Turag, Dhaleshwari and Shitalakshya rivers. It is also the largest Bengali-speaking city in the world. [...] As the capital of the People's Republic of Bangladesh, Dhaka is home to numerous state and diplomatic institutions. The Bangabhaban is the official residence and workplace of the President of Banglade

In [10]:
# cell: 6
 # LLM 

# Gemini Model
gemini_llm = None

if GOOGLE_API_KEY:
    gemini_llm = ChatGoogleGenerativeAI(
        model="gemini-3.8-flash",
        temperature=0,
        google_api_key=GOOGLE_API_KEY,
        max_retries=1,
        timeout=60
    )


# Ollama Local Model
ollama_llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0,
    base_url="http://localhost:11434"
)

print("Gemini and Ollama models configured!")


Gemini and Ollama models configured!


In [11]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a helpful AI assistant.

        Use the available tools whenever necessary.
        Analyze tool results carefully.
        Give a clear and accurate final answer.
        Do not repeat tool calls unnecessarily.
        """
    ),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

print("Native Tool Calling Prompt Created!")

Native Tool Calling Prompt Created!


In [12]:
# cell: 8
prompt

ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], input_types={'agent_scratchpad': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Ann

In [13]:
# cell: 9
#TOOLS

tools = [
    get_weather_data
]

if search_tool is not None:
    tools.append(search_tool)

print("Available Tools:")

for t in tools:
    print(t.name)

Available Tools:
get_weather_data
tavily_search_results_json


In [14]:
ollama_agent = create_agent(
    model=ollama_llm,
    tools=tools,
    system_prompt="""You are a helpful AI assistant.

    Use available tools when necessary.
    Analyze tool results carefully.
    Do not repeat completed tool calls.
    Always provide a clear final answer.
    """
)

print("Ollama Agent created successfully!")

ollama_executor = AgentExecutor(
    agent=ollama_agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=5,
    max_execution_time=120
)

print("Ollama Agent is ready!")

Ollama Agent created successfully!
Ollama Agent is ready!


In [15]:
questions = [
    "Find the capital of France",
    "What is the current weather in Paris?"
]

for question in questions:

    response = ollama_agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": question
            }
        ]
    })

    print("\nQuestion:", question)
    print("Answer:", response["messages"][-1].content)


Question: Find the capital of France
Answer: The capital of France is Paris.

Question: What is the current weather in Paris?
Answer: The current weather in Paris is overcast with a temperature of 3.6°C and a feels-like temperature of 0.7°C. The humidity is 84% and the wind speed is 6.2 km/h. There is no precipitation.


In [16]:
# cell: 13
print(response["messages"][-1].content)

The current weather in Paris is overcast with a temperature of 3.6°C and a feels-like temperature of 0.7°C. The humidity is 84% and the wind speed is 6.2 km/h. There is no precipitation.
